# Part 5: Fine-tuning with a loop you can read

**Fine-tuning** continues training a pretrained model on your own labelled data. You already know every line of the loop: this is the Module 07 loop with a pretrained body and a new head. Each cell below that trains takes about two minutes on a CPU.

In [1]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from review_data import as_lists, load_reviews, small_sample

reviews = load_reviews()
val_texts, val_labels = as_lists(reviews["validation"])

texts, labels = small_sample(reviews["train"], 200, seed=42)

c:\Users\student\AppData\Local\anaconda3\envs\aigc-5005-module09-hf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## What changes in the model

The body is pretrained. The two-layer head is new and starts with random weights, which is why Hugging Face prints a warning about newly initialized weights. That is expected.

In [2]:
from transformers import AutoModelForSequenceClassification

from finetune import MODEL, freeze_body

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
total = sum(p.numel() for p in model.parameters())
print("all parameters", total)
print("head only", freeze_body(model))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 555.59it/s]

all parameters 66955010
head only 592130


## Train only the head

The body is frozen, so a larger learning rate is safe.

In [3]:
from finetune import evaluate, fine_tune

model, tokenizer, history = fine_tune(
    texts, labels, val_texts, val_labels, head_only=True, lr=1e-3
)
for row in history:
    print(row)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 595.57it/s]


{'epoch': 1, 'train_loss': 0.6529, 'val_accuracy': 0.6407, 'val_macro_f1': 0.6135, 'seconds': 23.2}
{'epoch': 2, 'train_loss': 0.5413, 'val_accuracy': 0.6792, 'val_macro_f1': 0.6751, 'seconds': 32.7}
{'epoch': 3, 'train_loss': 0.5085, 'val_accuracy': 0.7073, 'val_macro_f1': 0.7071, 'seconds': 40.0}


## Train everything

Now the pretrained layers move too, so the learning rate must be small. A large one would wreck what the model already knows.

In [4]:
model, tokenizer, history = fine_tune(texts, labels, val_texts, val_labels, lr=5e-5)
for row in history:
    print(row)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 363.03it/s]


{'epoch': 1, 'train_loss': 0.6897, 'val_accuracy': 0.6848, 'val_macro_f1': 0.6775, 'seconds': 59.3}
{'epoch': 2, 'train_loss': 0.5806, 'val_accuracy': 0.6998, 'val_macro_f1': 0.6954, 'seconds': 57.6}
{'epoch': 3, 'train_loss': 0.4191, 'val_accuracy': 0.7083, 'val_macro_f1': 0.7051, 'seconds': 65.2}


In [5]:
print(evaluate(model, tokenizer, val_texts, val_labels))

{'accuracy': 0.7083, 'macro_f1': 0.7051}


## Try it

1. Train the whole model with a learning rate of `5e-3`. What happens to the validation accuracy?
2. Train for 6 epochs instead of 3. Does the training loss keep falling? Does the validation accuracy?
3. Train the head only for 3 epochs at `5e-5`. Why is it so much worse than at `1e-3`?